In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
from pathlib import Path # practice using this one as it's the modern way to work with path
from hydra.utils import get_original_cwd # didn't work in notebook

In [15]:
os.getcwd(), Path.cwd()

('d:\\a-study-on-transformer\\notebooks',
 WindowsPath('d:/a-study-on-transformer/notebooks'))

In [ ]:
from hydra import compose, initialize
from omegaconf import OmegaConf

In [5]:
with initialize(config_path="../config", version_base=None):
    cfg = compose(config_name="config")

In [8]:
print(OmegaConf.to_yaml(cfg))

params:
  epochs: 100
  lr: 0.003
  'n': 1000
  d_model: 10
  expansion: 4
  batch_size: 32
version: 0.0.1
tracking_uri: http://127.0.0.1:5000
experiment_name: deep-learning-experiment
run_name: test tag
stage: benchmark



In [11]:
print(print(cfg))

{'params': {'epochs': 100, 'lr': 0.003, 'n': 1000, 'd_model': 10, 'expansion': 4, 'batch_size': 32}, 'version': '0.0.1', 'tracking_uri': 'http://127.0.0.1:5000', 'experiment_name': 'deep-learning-experiment', 'run_name': 'test tag', 'stage': 'benchmark'}
None


In [12]:
print(OmegaConf.to_container(cfg.params, resolve=True))

{'epochs': 100, 'lr': 0.003, 'n': 1000, 'd_model': 10, 'expansion': 4, 'batch_size': 32}


In [16]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

In [20]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [191]:
class TestDataset(Dataset):
    def __init__(self, data):
        super().__init__()
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]


def test_collate_fn(batch, categorical_features, continuous_features):
    proxy = {}
    for feature in categorical_features:
        proxy[feature] = torch.stack([item[feature] for item in batch])
    for feature in continuous_features:
        proxy[feature] = torch.stack([item[feature] for item in batch]).unsqueeze(-1)
    # return {
    #     "cats": torch.stack([item["cats"] for item in batch]),
    #     "ages": torch.stack([item["ages"] for item in batch]).unsqueeze(-1),
    #     "prices": torch.stack([item["prices"] for item in batch]).unsqueeze(-1)
    # }
    return proxy

In [192]:
n = 1000
d_model = 512
max_len = 5
vocab_size = 5

In [211]:
from dataclasses import dataclass, field
from typing import List, Union

@dataclass
class CategoricalFeature:
    name: str
    d_model: int
    max_len: int
    vocab_size: int

@dataclass
class ContinuousFeature:
    name: str
    d_model: int
    max_len: int

@dataclass
class FeatureConfig:
    features:List[Union[CategoricalFeature, ContinuousFeature]]
    categorical_features:List[str] = field(init=False)
    continuous_features:List[str] = field(init=False)

    def __post_init__(self):
        self.categorical_features = [f.name for f in self.features if isinstance(f, CategoricalFeature)]
        self.continuous_features = [f.name for f in self.features if isinstance(f, ContinuousFeature)]

features=FeatureConfig(
    features=[
        # CategoricalFeature(name="cats", d_model=d_model, max_len=max_len, vocab_size=vocab_size+1),
        ContinuousFeature(name="ages", d_model=d_model, max_len=max_len),
        # ContinuousFeature(name="prices", d_model=d_model, max_len=max_len),
    ]
)
features.features, features.categorical_features, features.continuous_features

([ContinuousFeature(name='ages', d_model=512, max_len=5)], [], ['ages'])

In [212]:
from functools import partial
torch.manual_seed(42)

cats  = torch.randint(1, vocab_size+1,  size=(n, max_len))
ages   = torch.randint(21, 79, size=(n, max_len)).float()
prices = torch.rand(n, max_len) * (30000 - 2500) + 2500

samples = [
    {"cats": i, "ages": a, "prices": p}
    for i, a, p in zip(cats, ages, prices)
]
batch_size = 32
partial_collate_fn = partial(test_collate_fn, categorical_features=features.categorical_features, continuous_features=features.continuous_features)
dataset = TestDataset(data=samples)
loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, collate_fn=partial_collate_fn)
batch = next(iter(loader))

In [213]:
batch.keys()

dict_keys(['ages'])

In [214]:
from typing import Dict
from enum import StrEnum

class FusionStrategy(StrEnum):
    SUM = "sum"
    CONCAT = "concat"
    SUM_CONCAT = "sum_concat"

class MockFusionEmbedding(nn.Module):
    def __init__(self, features:FeatureConfig, strategy:FusionStrategy):
        super().__init__()
        self.features = features
        self.emb_modules = nn.ModuleDict(self.create_emb_module(features))
        self.strategy = strategy
        self.initialize_strategy(strategy, features)

    def initialize_strategy(self, strategy:FusionStrategy, features:FeatureConfig):
        d_model = features.features[0].d_model
        concat_dim = 0
        if strategy == FusionStrategy.CONCAT:
            concat_dim += d_model * len(features.features)
        elif strategy == FusionStrategy.SUM_CONCAT:
            concat_dim += d_model * len({type(feature) for feature in features.features})
        self.out_proj = nn.Linear(concat_dim, d_model)

    def create_emb_module(self, features:FeatureConfig):
        feature_dict = {}
        for feature in features.features:
            if isinstance(feature, CategoricalFeature):
                feature_dict[feature.name] = nn.Embedding(
                    num_embeddings=feature.vocab_size,
                    embedding_dim=feature.d_model
                )
            elif isinstance(feature, ContinuousFeature):
                feature_dict[feature.name] = nn.Linear(
                    in_features=1,
                    out_features=feature.d_model
                )
            else:
                raise TypeError(f"{type(feature)} is not implemented, Try: {CategoricalFeature} or {ContinuousFeature}")
        return feature_dict

    def run_strategy(self, results:Dict[str, torch.Tensor], features:FeatureConfig)->torch.Tensor:
        if self.strategy == FusionStrategy.CONCAT:
            result = torch.cat(list(results.values()), dim=-1)
            result = self.out_proj(result)
            return result
        if self.strategy == FusionStrategy.SUM_CONCAT:
            feature_vectors = []
            if len(features.categorical_features)>0:
                feature_vectors.append(
                    torch.stack([results[feature] for feature in features.categorical_features], dim=0).sum(dim=0)
                )
            if len(features.continuous_features)>0:
                feature_vectors.append(
                    torch.stack([results[feature] for feature in features.continuous_features], dim=0).sum(dim=0)
                )
            result = torch.cat(feature_vectors, dim=-1)
            result = self.out_proj(result)
            return result
        # if not in concat or sum_concat
        result = torch.stack(list(results.values()), dim=0).sum(dim=0)
        return result

    def forward(self, X:Dict[str, torch.Tensor], features:FeatureConfig)->torch.Tensor:
        """
        Args:
            X (dict) : a batch of dictionary with key as a feature name
            features (FeatureConfig) : a feature config

        Returns:
            torch.Tensor : embedded features
        """
        results = {}
        for feature in features.features:
            emb = self.emb_modules[feature.name](X[feature.name])
            results[feature.name] = emb
        return self.run_strategy(results, self.features)

In [217]:
for fusion_stragegy in ["sum", "concat", "sum_concat"]:
    fusion_stragegy = FusionStrategy(fusion_stragegy)
    fusion = MockFusionEmbedding(
        features=features,
        strategy=fusion_stragegy
    )
    result = fusion(X=batch, features=features)
    print(fusion_stragegy, batch.keys(), batch[list(batch.keys())[0]].shape, result.shape)

sum dict_keys(['ages']) torch.Size([32, 5, 1]) torch.Size([32, 5, 512])
concat dict_keys(['ages']) torch.Size([32, 5, 1]) torch.Size([32, 5, 512])
sum_concat dict_keys(['ages']) torch.Size([32, 5, 1]) torch.Size([32, 5, 512])
